In [2]:
from llm import classify_email

classify_email('hi')

'Hello! How can I assist you today?'

In [3]:
email = {
    "subject": "Can't login ",
    "body"   : "Our entire team can't login.  paid for annual "
               "plan last week. Board demo in 3 hours.entire team cant login",
    "correct": "Billing"
}


subject = email['subject']
body= email['body']


PROMPT_CURRENT = f"""You are an expert support email classifier.

Classify into EXACTLY ONE:
- Billing
- Technical
- Feature Request
- Spam
- Other

Subject : {subject}
Body    : {body}

Return ONLY: Category | Confidence (1-10)"""

In [4]:
output = classify_email(PROMPT_CURRENT)

In [5]:
output

'Technical | 9'

In [ ]:
# zero shot cot prompting 
PROMPT_COT_V2 = f"""You are an expert support email classifier
for a B2B SaaS company.

Before classifying, think step by step:
1. What is the surface complaint?
2. What is the ROOT CAUSE behind it?
3. Which team OWNS this problem?
4. What is the business impact?

Then classify into EXACTLY ONE:
- Billing
- Technical
- Feature Request
- Spam
- Other


Subject : {subject}
Body    : {body}

Important:
- Technical team fixes bugs and crashes
- Billing team fixes payment and account activation issues
- Surface complaint does not decide the category
- ROOT CAUSE decides the category

Format EXACTLY like this:
THINKING:
1. Surface complaint: ...
2. Root cause: ...
3. Team that OWNS this: ...
4. Business impact: ...

CATEGORY: ..."""


In [7]:
result = classify_email(PROMPT_COT_V2)
print(result)

THINKING:
1. Surface complaint: The team cannot log in to the system.
2. Root cause: Payment and account activation issue despite having paid for the annual plan.
3. Team that OWNS this: Billing team.
4. Business impact: The inability to log in impacts a board demo scheduled in 3 hours, which could harm client satisfaction and business reputation.

CATEGORY: Billing


In [ ]:
PROMPT_COT_V2 = f"""You are an expert support email classifier
for a B2B SaaS company.

Before classifying, think step by step:
1. What is the surface complaint?
2. What is the ROOT CAUSE behind it?
3. Which team OWNS this problem?
4. What is the business impact?

Then classify into EXACTLY ONE:
- Billing
- Technical
- Feature Request
- Spam
- Other

See the below examples

EXAMPLE 1:
Subject: NACH mandate registration failed
Body: Our NACH mandate for auto-debit failed. Customer accounts 
cannot be debited for EMI this month.

THINKING:
- Surface complaint: NACH mandate registration failure.
- Root cause: NACH = National Automated Clearing House, a payment 
  collection instrument. Failure = payment collection blocked for 
  all customers using this mandate. This is a payment infrastructure issue.
- Team that should act: Payments/Billing team — specifically the 
  payment gateway integration team.
- Business impact: Multiple customer EMIs will fail today. Revenue 
  loss + customer escalations expected within hours.

CLASSIFICATION:
{"category": "Transaction Failed", "urgency": "Critical"}

---

EXAMPLE 2:
Subject: UPI reversal not credited back
Body: I did a UPI payment 3 days ago. It failed but the amount 
was deducted. Not refunded yet.

THINKING:
- Surface complaint: Payment deducted but service not received.
- Root cause: UPI reversal (failed transaction refund) not processed 
  within standard T+3 timeline. This is a settlement/reconciliation issue.
- Team that should act: Finance/Settlement team — check bank reconciliation 
  and trigger manual refund if auto-reversal stuck.
- Business impact: Customer's money held for 3+ days = regulatory risk + 
  customer churn if not resolved today.

CLASSIFICATION:
{"category": "Settlement Delay", "urgency": "High"}

---

EXAMPLE 3:  
Subject: KYC documents rejected again
Body: Our KYC was rejected for the 3rd time. We submitted Aadhaar, 
PAN, and GST certificate. What else do you need?

THINKING:
- Surface complaint: KYC documents repeatedly rejected.
- Root cause: Either unclear document requirements OR automated KYC 
  system mis-rejecting valid documents. Customer frustration high.
- Team that should act: KYC/Compliance team — needs manual review, 
  and possibly update to rejection reason communication.
- Business impact: Customer cannot go live. Blocking revenue for both 
  parties. Risk of customer choosing competitor.

CLASSIFICATION:
{"category": "KYC Issue", "urgency": "High"}



Subject : {subject}
Body    : {body}

Important:
- Technical team fixes bugs and crashes
- Billing team fixes payment and account activation issues
- Surface complaint does not decide the category
- ROOT CAUSE decides the category

Format EXACTLY like this:
THINKING:
1. Surface complaint: ...
2. Root cause: ...
3. Team that OWNS this: ...
4. Business impact: ...

CATEGORY: ..."""

## Auto COT

In [8]:
warmup_prompt = """You are a email support analyst. 
    
Before I give you a real email to classify, I want you to demonstrate 
your reasoning ability. Generate 3 different example emails a Saas crm 
support team might receive, and show step-by-step how you would 
classify each one. 

Cover different categories: 
- Billing
- Technical
- Feature Request
- Spam
- Other

Format each as:
EXAMPLE [N]:
Email: [email text you invent]
Reasoning: [step-by-step thinking]
Classification: [category + urgency]"""


In [9]:
self_example = classify_email(warmup_prompt)

In [16]:
# print(self_example)

email = {
    "subject": "Can't login ",
    "body"   : "Our entire team can't login.  paid for annual "
               "plan last week. Board demo in 3 hours.entire team cant login",
    "correct": "Billing"
}

subject = email['subject']
body= email['body']


classify_prompt = f"""You are a fintech support analyst.

You just demonstrated your reasoning ability with these examples:
{self_example}

Now apply the SAME reasoning pattern to this real email:

Subject: {subject}
Body: {body}

Important:
- Technical team fixes bugs and crashes
- Billing team fixes payment and account activation issues
- Surface complaint does not decide the category
- ROOT CAUSE decides the category


Follow the same format: Reasoning → Classification JSON."""

In [17]:
# print(classify_prompt)

result = classify_email(classify_prompt)
print(result)

**Email:**
Subject: Can't login

Our entire team can't login. Paid for annual plan last week. Board demo in 3 hours. Entire team can't login.

**Reasoning:**
1. **Read the email** to understand the core issue: The sender's entire team is unable to log in.
2. **Identify keywords**: "entire team can't login," "paid for annual plan," "board demo in 3 hours."
3. **Determine the impact**: The inability to log in affects the entire team and is critical because a board demo is scheduled in 3 hours.
4. **Assess urgency**: The mention of a "board demo in 3 hours" indicates a high urgency since it impacts a crucial meeting.
5. **Root Cause Analysis**: The mention of "paid for annual plan last week" suggests the problem may not be purely technical but potentially related to account activation or billing.
6. **Classification Decision**: Although logging in is a technical issue, the root cause likely involves payment recognition or account activation timing, which aligns with billing issues.

**Cla

## Self consistency

In [18]:
# prompt =# zero shot cot prompting 
PROMPT_COT_V2 = f"""You are an expert support email classifier
for a B2B SaaS company.

Before classifying, think step by step:
1. What is the surface complaint?
2. What is the ROOT CAUSE behind it?
3. Which team OWNS this problem?
4. What is the business impact?

Then classify into EXACTLY ONE:
- Billing
- Technical
- Feature Request
- Spam
- Other


Subject : {subject}
Body    : {body}

Important:
- Technical team fixes bugs and crashes
- Billing team fixes payment and account activation issues
- Surface complaint does not decide the category
- ROOT CAUSE decides the category

Format EXACTLY like this:
THINKING:
1. Surface complaint: ...
2. Root cause: ...
3. Team that OWNS this: ...
4. Business impact: ...

CATEGORY: ..."""



for i in range(5):
    result = classify_email(PROMPT_COT_V2)
    print(f"Run {i+1}:")
    print(result)
    print("\n")


Run 1:
THINKING:
1. Surface complaint: The user's entire team is unable to log in.
2. Root cause: The issue could be related to account activation or verification following a recent payment for the annual plan.
3. Team that OWNS this: Billing team.
4. Business impact: High impact on the customer's business as they have an important board demo scheduled in 3 hours.

CATEGORY: Billing


Run 2:
THINKING:
1. Surface complaint: The entire team can't log in, and they have a board demo in 3 hours.
2. Root cause: Potential account activation issue due to recent payment; need to verify payment was processed and account status is active.
3. Team that OWNS this: Billing team, as it involves verifying payment and account activation.
4. Business impact: High, as the inability to log in could affect an important board demo and overall client usage.

CATEGORY: Billing


Run 3:
THINKING:
1. Surface complaint: The entire team can't log in.
2. Root cause: Likely an account activation issue related to th

## TOT

In [20]:
prompt = f"""You are a senior support manager at a B2B SaaS company.
A customer sent this email and it was classified as:
Category: 'category':'billing','urgency':'high'

Customer email:
Subject: {email['subject']}
Body: {email['body']}

Your task: Draft a response. But first, explore 3 DIFFERENT response strategies.
For each strategy, write the draft AND score it.

STRATEGY 1: Immediate Action
- Draft: [Write the response using this approach]
- Score: X/10 — reason why this score

STRATEGY 2: Empathetic + Diagnostic  
- Draft: [Write the response using this approach]
- Score: X/10 — reason why this score

STRATEGY 3: Escalation + Interim Relief
- Draft: [Write the response using this approach]  
- Score: X/10 — reason why this score

SELECTED STRATEGY: [Which strategy number and why]

FINAL RESPONSE:
[The winning response — may incorporate elements from multiple strategies]"""



output = classify_email(prompt)

In [22]:
print(output)

**STRATEGY 1: Immediate Action**

- **Draft:**
  Subject: Immediate Assistance for Login Issue

  Hi [Customer's Name],

  We understand the urgency of your situation. We are actively working to resolve the login issue. Please bear with us for the next few minutes as we reset your team's access.

  If your team still can't login after 15 minutes, please contact me directly at [your phone number]. Rest assured, we will do everything in our power to resolve this before your board demo.

  Thank you for your patience and understanding.

  Best regards,  
  [Your Name]  
  Senior Support Manager

- **Score: 7/10** — This response is effective in communicating immediate action and offers a direct line to support, but it lacks a deeper exploration of the possible account/payment issues that could be causing the login problem.

**STRATEGY 2: Empathetic + Diagnostic**

- **Draft:**
  Subject: We're Here to Help with Your Login Issue

  Hi [Customer's Name],

  I'm really sorry to hear that you